> **한눈에 보기**
> - LangChain의 `ChatOpenAI`로 질문하기 (교재 92쪽)
> - `invoke()`는 `AIMessage`를 돌려준다: `.content`(답), `.response_metadata`(모델, 토큰 사용량 등)
> - `.bind(logprobs=True)`로 토큰마다 모델의 확신 정도(로그 확률)를 함께 받는다
> - `stream()`은 답을 조각(chunk)으로 받는다 → 채팅처럼 바로바로 출력
> - 정리 문서: [OpenAI가이드.md](OpenAI가이드.md)

### 1. ChatOpenAI로 질문하기

In [1]:
# LangChain의 OpenAI 채팅 모델 (pip install langchain-openai)
from langchain_openai import ChatOpenAI

In [7]:
# temperature=0.1: 낮을수록 같은 질문에 비슷한 답
llm = ChatOpenAI(temperature=0.1, model="gpt-5.6-luna")

In [8]:
# AIMessage를 그대로 f-string에 넣으면 content뿐 아니라 메타데이터까지 전부 찍힌다
question = "대한민국의 수도는 어디인가요?" 

print(f"[답변]: {llm.invoke(question)}")

[답변]: content='대한민국의 수도는 서울특별시입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 15, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EMqfsXhpT2Dcq5widpWVTadd6U22d', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a08f87-4a4d-7211-af1f-b65e2d298b50-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 14, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


### 2. 응답 객체 살펴보기

In [9]:
# 결과를 변수에 받아 두고 하나씩 꺼내 보기
response = llm.invoke(question)
response

AIMessage(content='대한민국의 수도는 서울특별시입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 15, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EMqh3qKC9k10tUsa9ejaBPUPJyJe0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a08f88-6e28-7a90-9ba8-98716e0c1910-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 14, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [10]:
# 답 글자만
response.content

'대한민국의 수도는 서울특별시입니다.'

In [11]:
# 모델 이름, 토큰 사용량 등 부가 정보
response.response_metadata

{'token_usage': {'completion_tokens': 14,
  'prompt_tokens': 15,
  'total_tokens': 29,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0,
   'text_tokens': None},
  'prompt_tokens_details': {'audio_tokens': 0,
   'cache_write_tokens': 0,
   'cached_tokens': 0,
   'image_tokens': None,
   'text_tokens': None}},
 'model_provider': 'openai',
 'model_name': 'gpt-5.6-luna',
 'system_fingerprint': None,
 'id': 'chatcmpl-EMqh3qKC9k10tUsa9ejaBPUPJyJe0',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

In [12]:
# prompt_tokens(입력) + completion_tokens(출력) = total_tokens
response.response_metadata["token_usage"]

{'completion_tokens': 14,
 'prompt_tokens': 15,
 'total_tokens': 29,
 'completion_tokens_details': {'accepted_prediction_tokens': 0,
  'audio_tokens': 0,
  'reasoning_tokens': 0,
  'rejected_prediction_tokens': 0,
  'text_tokens': None},
 'prompt_tokens_details': {'audio_tokens': 0,
  'cache_write_tokens': 0,
  'cached_tokens': 0,
  'image_tokens': None,
  'text_tokens': None}}

### 3. logprobs: 토큰별 확신 정도

In [16]:
# 참고: 여기서는 model_name='gpt-4o-mini' (위의 llm과 다른 모델, model과 model_name은 같은 설정)
# .bind(logprobs=True): 호출할 때마다 logprobs=True 옵션이 붙는 모델
llm_with_logprob = ChatOpenAI(
    temperature=0.1,
    max_tokens=2048,
    model_name="gpt-4o-mini",
).bind(logprobs=True)

In [17]:
# response_metadata['logprobs']에 토큰마다 logprob이 들어 있다 (0에 가까울수록 확신이 큼)
question = "대한민국의 수도는 어디인가요?"

response = llm_with_logprob.invoke(question)
response.response_metadata

{'token_usage': {'completion_tokens': 8,
  'prompt_tokens': 16,
  'total_tokens': 24,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0,
   'text_tokens': None},
  'prompt_tokens_details': {'audio_tokens': 0,
   'cache_write_tokens': None,
   'cached_tokens': 0,
   'image_tokens': None,
   'text_tokens': None}},
 'model_provider': 'openai',
 'model_name': 'gpt-4o-mini-2024-07-18',
 'system_fingerprint': 'fp_4adbad45a5',
 'id': 'chatcmpl-EMqoiLb8K2yUrNyQ6JLfkGrxx94yJ',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': {'content': [{'token': '대한',
    'bytes': [235, 140, 128, 237, 149, 156],
    'logprob': -4.310693111619912e-05,
    'top_logprobs': []},
   {'token': '민국',
    'bytes': [235, 175, 188, 234, 181, 173],
    'logprob': -5.512236498361744e-07,
    'top_logprobs': []},
   {'token': '의',
    'bytes': [236, 157, 152],
    'logprob': -2.7968066206085496e-05,
    'top

### 4. stream: 조각씩 받기

In [18]:
# stream: 만들어지는 대로 조각(AIMessageChunk)을 하나씩 돌려준다
# end='': 줄바꿈 없이 이어 찍기 / flush=True: 바로 화면에 내보내기
answer = llm.stream("대한민국의 아름다운 관광지 10곳과 주소를 알려주세요!")

for token in answer:
    print(token.content, end="", flush=True)

대한민국의 아름다운 관광지 10곳과 대표 주소를 소개합니다.  
※ 넓은 관광지는 주차장·입구 기준의 대표 주소입니다.

| 관광지 | 지역 | 주소 |
|---|---|---|
| 1. 경복궁 | 서울 종로구 | 서울특별시 종로구 사직로 161 |
| 2. 남산서울타워 | 서울 용산구 | 서울특별시 용산구 남산공원길 105 |
| 3. 해운대해수욕장 | 부산 해운대구 | 부산광역시 해운대구 해운대해변로 264 |
| 4. 설악산국립공원 | 강원특별자치도 속초시 | 강원특별자치도 속초시 설악산로 833 |
| 5. 경주 불국사 | 경상북도 경주시 | 경상북도 경주시 불국로 385 |
| 6. 성산일출봉 | 제주특별자치도 서귀포시 | 제주특별자치도 서귀포시 성산읍 일출로 284-12 |
| 7. 전주한옥마을 | 전북특별자치도 전주시 | 전북특별자치도 전주시 완산구 기린대로 99 |
| 8. 순천만국가정원 | 전라남도 순천시 | 전라남도 순천시 국가정원1호길 47 |
| 9. 담양 죽녹원 | 전라남도 담양군 | 전라남도 담양군 담양읍 죽녹원로 119 |
| 10. 안동 하회마을 | 경상북도 안동시 | 경상북도 안동시 풍천면 하회종가길 2-1 |

특히 **성산일출봉의 일출**, **설악산의 단풍**, **순천만국가정원의 계절별 꽃**, **담양 죽녹원의 대나무숲**은 계절에 따라 더욱 아름답습니다.

In [19]:
# 조각의 content를 이어 붙여 전체 답 만들기
answer = llm.stream("대한민국의 아름다운 관광지 10곳과 주소를 알려주세요!")

final_answer = "" 
for token in answer:
    final_answer += token.content

In [20]:
# 같은 질문이어도 위 결과와 목록이 다르다 - 답은 매번 조금씩 달라질 수 있다
print(final_answer)

대한민국에서 자연과 역사, 바다의 아름다움을 함께 느낄 수 있는 관광지 10곳을 소개합니다. 주소는 방문자센터나 대표 입구 기준입니다.

| 관광지 | 지역 | 주소 |
|---|---|---|
| 1. 성산일출봉 | 제주특별자치도 | 제주특별자치도 서귀포시 성산읍 일출로 284-12 |
| 2. 경복궁 | 서울 | 서울특별시 종로구 사직로 161 |
| 3. 남이섬 | 강원특별자치도 | 강원특별자치도 춘천시 남산면 남이섬길 1 |
| 4. 해운대해수욕장 | 부산 | 부산광역시 해운대구 해운대해변로 264 |
| 5. 불국사 | 경상북도 경주 | 경상북도 경주시 불국로 385 |
| 6. 순천만습지 | 전라남도 순천 | 전라남도 순천시 순천만길 513-25 |
| 7. 죽녹원 | 전라남도 담양 | 전라남도 담양군 담양읍 죽녹원로 119 |
| 8. 설악산국립공원 | 강원특별자치도 속초 | 강원특별자치도 속초시 설악산로 1091 |
| 9. 안동 하회마을 | 경상북도 안동 | 경상북도 안동시 풍천면 하회종가길 2-1 |
| 10. 오동도 | 전라남도 여수 | 전라남도 여수시 오동도로 222 |

간단한 추천 포인트:

- **일출 감상:** 성산일출봉  
- **궁궐과 전통문화:** 경복궁, 안동 하회마을  
- **단풍과 산행:** 설악산국립공원  
- **대나무숲 산책:** 담양 죽녹원  
- **갈대와 생태 여행:** 순천만습지  
- **바다 풍경:** 해운대해수욕장, 오동도  
- **역사 유적:** 불국사  
- **호수와 낭만적인 산책:** 남이섬  

방문 전에는 운영시간, 입장료, 주차 가능 여부와 계절별 통제 정보를 공식 홈페이지에서 확인하는 것이 좋습니다.


## 🔁 복습 질문

<details>
<summary>1. <code>llm.invoke(question)</code>의 결과에서 답 글자만 꺼내려면?</summary>

`response.content`. 결과는 `AIMessage` 객체라서 그대로 출력하면 메타데이터까지 나온다.
</details>

<details>
<summary>2. 토큰 사용량은 어디서 보나?</summary>

`response.response_metadata["token_usage"]` (`prompt_tokens`, `completion_tokens`, `total_tokens`).
</details>

<details>
<summary>3. logprob이 -0.0002인 토큰과 -2.5인 토큰 중 모델이 더 확신한 것은?</summary>

-0.0002. 로그 확률은 0에 가까울수록 확률이 1에 가깝다는 뜻이다.
</details>

<details>
<summary>4. <code>stream</code>의 조각을 모아 전체 답을 만드는 방법은?</summary>

`final_answer = ""`로 시작해 `for token in answer: final_answer += token.content`.
</details>
